# Shallow water on the sphere: families, forcings, diagnostics and ensembles

`SphericalShallowWater` solves the shallow water equations on the rotating sphere with spherical harmonics. This notebook runs every initial-condition family and every stochastic forcing at a small size, and shows the diagnostics, a reference ensemble, the coarse-grained truth and the rotation operator.

Everything runs at the band limit `lmax = 43` on a 64 x 128 Gauss-Legendre grid (about 2.8 degrees), in about fifteen minutes on one GPU. The relaxation, drag and forcing values are those of the benchmark datasets, which are the configs `shallow_water_sphere_d*.yaml` and `shallow_water_sphere_t*.yaml`.

The saved channels are `h` (height of the free surface above the mean depth, in m), `u_phi` (eastward wind) and `u_theta` (southward wind), both in m/s.

In [ ]:
import math
import os

import matplotlib.pyplot as plt
import torch

from autosim.experimental.simulations.shallow_water_sphere import (
    SphericalShallowWater,
    fields_to_spectral,
    height_error_norms,
    height_power_spectrum,
    integrate_trajectories,
    invariants,
    kinetic_energy_spectrum,
    non_zonal_energy,
    rotate_coefficients,
    rotate_fields,
    spectral_to_fields,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs("figures", exist_ok=True)

lmax = 43
nlat = 64
nlon = 128
radius = 6.37122e6  # [m]
gravity = 9.80616  # [m/s2]
williamson_depth = 2.94e4 / gravity  # h_0 of W92 case 2 [m]

## Initial-condition families

`initial_condition` selects the family, and `parameters_range` gives the bounds of its parameters. Equal bounds fix a parameter, so the ranges below give the published case of each family:

- `galewsky`: unstable jet with a height bump (Galewsky et al. 2004);
- `galewsky_unperturbed`: the same jet alone, which must stay zonal;
- `williamson2`: steady flow about an axis tilted by `tilt` (W92 case 2);
- `rossby_haurwitz`: wavenumber-4 wave (W92 case 6);
- `mountain`: zonal flow over an isolated mountain (W92 case 5);
- `random_balanced`: random flow with its balanced height.

Each run lasts 6 days and is saved every 24 h.

In [ ]:
simulators = {}

simulators["galewsky"] = SphericalShallowWater(
    lmax=lmax, nlat=nlat, nlon=nlon,
    T=144.0, dt_save=24.0, skip_nt=0,
    batch_size=1, device=device, log_level="warning",
    initial_condition="galewsky",
    mean_depth=1.0e4,
    parameters_range={
        "jet_speed": (80.0, 80.0),
        "height_perturbation_m": (120.0, 120.0),
        "perturbation_lon": (math.pi, math.pi),
        "perturbation_lat": (math.pi / 4, math.pi / 4),
    },
)

simulators["galewsky_unperturbed"] = SphericalShallowWater(
    lmax=lmax, nlat=nlat, nlon=nlon,
    T=144.0, dt_save=24.0, skip_nt=0,
    batch_size=1, device=device, log_level="warning",
    initial_condition="galewsky_unperturbed",
    mean_depth=1.0e4,
    parameters_range={"jet_speed": (80.0, 80.0)},
)

simulators["williamson2"] = SphericalShallowWater(
    lmax=lmax, nlat=nlat, nlon=nlon,
    T=144.0, dt_save=24.0, skip_nt=0,
    batch_size=1, device=device, log_level="warning",
    initial_condition="williamson2",
    mean_depth=williamson_depth,
    parameters_range={"flow_speed": (38.61, 38.61), "tilt": (0.8, 0.8)},
)

simulators["rossby_haurwitz"] = SphericalShallowWater(
    lmax=lmax, nlat=nlat, nlon=nlon,
    T=144.0, dt_save=24.0, skip_nt=0,
    batch_size=1, device=device, log_level="warning",
    initial_condition="rossby_haurwitz",
    mean_depth=8000.0,
    parameters_range={"wavenumber": (4.0, 4.0), "wave_amplitude": (7.848e-6, 7.848e-6), "mean_depth": (8000.0, 8000.0)},
)

simulators["mountain"] = SphericalShallowWater(
    lmax=lmax, nlat=nlat, nlon=nlon,
    T=144.0, dt_save=24.0, skip_nt=0,
    batch_size=1, device=device, log_level="warning",
    initial_condition="mountain",
    mean_depth=5960.0,
    parameters_range={
        "flow_speed": (20.0, 20.0),
        "mountain_height": (2000.0, 2000.0),
        "mountain_radius": (math.pi / 9, math.pi / 9),
        "mountain_lon": (1.5 * math.pi, 1.5 * math.pi),
        "mountain_lat": (math.pi / 6, math.pi / 6),
    },
    return_constant_fields=True,
)

simulators["random_balanced"] = SphericalShallowWater(
    lmax=lmax, nlat=nlat, nlon=nlon,
    T=144.0, dt_save=24.0, skip_nt=0,
    batch_size=1, device=device, log_level="warning",
    initial_condition="random_balanced",
    mean_depth=1.0e4,
    parameters_range={"amp": (20.0, 20.0), "initial_wavenumber": (6.0, 6.0), "initial_bandwidth": (2.0, 2.0)},
)

family_runs = {}
for family in simulators:
    family_runs[family] = simulators[family].forward_samples_spatiotemporal(n=1, random_seed=0)
    print(family, tuple(family_runs[family]["data"].shape))

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(24, 5))
for column, family in enumerate(simulators):
    sim = simulators[family]
    last_frame = family_runs[family]["data"][:, -1]
    state = fields_to_spectral(sim.dynamics, last_frame)
    vorticity = sim.dynamics.to_grid(state[:, 1])[0].cpu()
    height = last_frame[0, :, :, 0]

    limit = vorticity.abs().max().item()
    axes[0, column].imshow(vorticity, extent=[0, 360, -90, 90], cmap="RdBu_r", vmin=-limit, vmax=limit, aspect="auto")
    axes[0, column].set_title(family + "\nvorticity [1/s], day 6")
    axes[1, column].imshow(height, extent=[0, 360, -90, 90], cmap="viridis", aspect="auto")
    axes[1, column].set_title("h [m], day 6")
    axes[1, column].set_xlabel("longitude [deg]")
axes[0, 0].set_ylabel("latitude [deg]")
axes[1, 0].set_ylabel("latitude [deg]")
fig.tight_layout()
fig.savefig("figures/benchmark_families.png", dpi=110)
plt.show()

## Spectra

`fields_to_spectral` converts saved frames back to spherical-harmonic coefficients. `height_power_spectrum` gives the power of the depth per degree $l$. `kinetic_energy_spectrum` gives the kinetic energy per degree, split into a rotational part (from the vorticity) and a divergent part (from the divergence). Their sum over the degrees is the global mean of $\tfrac12|\mathbf{u}|^2$.

In [ ]:
height_spectra = {}
rotational_spectra = {}
divergent_spectra = {}
for family in simulators:
    sim = simulators[family]
    last_frame = family_runs[family]["data"][:, -1]
    state = fields_to_spectral(sim.dynamics, last_frame)

    height_spectrum = height_power_spectrum(state, gravity)  # (1, lmax)
    kinetic_spectrum = kinetic_energy_spectrum(state, radius)  # (1, 2, lmax): rotational, divergent
    height_spectra[family] = height_spectrum[0].cpu()
    rotational_spectra[family] = kinetic_spectrum[0, 0].cpu()
    divergent_spectra[family] = kinetic_spectrum[0, 1].cpu()

    rotational_energy = rotational_spectra[family].sum().item()
    divergent_energy = divergent_spectra[family].sum().item()
    mean_kinetic_energy = rotational_energy + divergent_energy
    divergent_share = divergent_energy / mean_kinetic_energy
    print(f"{family:22s} mean kinetic energy {mean_kinetic_energy:8.2f} m2/s2, divergent share {divergent_share:.1e}")

In [ ]:
degrees = torch.arange(lmax)
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for family in simulators:
    axes[0].loglog(degrees[1:], height_spectra[family][1:], label=family)
    axes[1].loglog(degrees[1:], rotational_spectra[family][1:], label=family)
    axes[2].loglog(degrees[1:], divergent_spectra[family][1:].clamp(min=1e-30), label=family)
axes[0].set_title("power of the depth [m2]")
axes[1].set_title("rotational kinetic energy [m2/s2]")
axes[2].set_title("divergent kinetic energy [m2/s2]")
for axis in axes:
    axis.set_xlabel("degree l")
    axis.set_ylim(bottom=1e-16)
axes[0].legend()
fig.tight_layout()
fig.savefig("figures/benchmark_spectra.png", dpi=110)
plt.show()

## Invariants and error norms

`invariants` integrates the mass, the total energy and the potential enstrophy over the sphere. Without forcing, the hyperdiffusion is the only sink, so the mass is constant and the two others decrease slowly. W92 normalises each invariant as $(I(t) - I(0)) / I(0)$.

W92 case 2 has an analytic steady solution. Its error after 5 days is measured with the W92 norm $l_2(h)$ at three resolutions. The basis represents this smooth state exactly, so the error is the hyperdiffusion acting on the wind, and it decreases as $((L-1)L)^{-4}$. These runs use the solver functions directly, in double precision, since the saved frames are single precision.

In [ ]:
# Invariants along the Galewsky run, from its saved frames
sim = simulators["galewsky"]
frames = family_runs["galewsky"]["data"][0]
invariant_series = []
for frame in range(frames.shape[0]):
    state = fields_to_spectral(sim.dynamics, frames[frame : frame + 1])
    invariant_series.append(invariants(sim.dynamics, state)[0].cpu())
invariant_series = torch.stack(invariant_series)
relative_change = (invariant_series - invariant_series[0]) / invariant_series[0]
mass_drift = relative_change[-1, 0].item()
energy_drift = relative_change[-1, 1].item()
enstrophy_drift = relative_change[-1, 2].item()
print(f"relative change at day 6: mass {mass_drift:.1e}, energy {energy_drift:.1e}, potential enstrophy {enstrophy_drift:.1e}")

# W92 case 2 at tilt pi/2: error after 5 days at three resolutions, in double precision
convergence_lmax = [11, 22, 43]
convergence_nlat = [16, 32, 64]
convergence_nlon = [32, 64, 128]
convergence_errors = []
for index in range(3):
    resolution = SphericalShallowWater(
        lmax=convergence_lmax[index],
        nlat=convergence_nlat[index],
        nlon=convergence_nlon[index],
        T=120.0, dt_save=24.0, skip_nt=0,
        batch_size=1, device=device, log_level="warning",
        initial_condition="williamson2",
        mean_depth=williamson_depth,
        parameters_range={"flow_speed": (38.61, 38.61), "tilt": (math.pi / 2, math.pi / 2)},
    )
    dynamics = resolution.dynamics
    rows = resolution.sample_inputs(1, random_seed=0)
    state, coriolis, surface_geopotential = resolution.initial_problem(rows, noise_generators=[])
    output = integrate_trajectories(
        dynamics,
        state,
        n_saves=5,
        steps_per_save=resolution.steps_per_save,
        skip_nt=0,
        coriolis=coriolis,
    )
    fields_start = spectral_to_fields(dynamics, state)
    fields_end = spectral_to_fields(dynamics, output["final_state"])
    height_start = fields_start[..., 0] + williamson_depth
    height_end = fields_end[..., 0] + williamson_depth
    l1, l2, linf = height_error_norms(height_end, height_start, dynamics.quad_weights)
    convergence_errors.append(l2.item())
    print(f"lmax = {convergence_lmax[index]:3d}: l2(h) after 5 days = {l2.item():.2e}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
days = torch.arange(relative_change.shape[0])
axes[0].plot(days, relative_change[:, 0], label="mass")
axes[0].plot(days, relative_change[:, 1], label="energy")
axes[0].plot(days, relative_change[:, 2], label="potential enstrophy")
axes[0].set_xlabel("day")
axes[0].set_title("Galewsky jet: (I(t) - I(0)) / I(0)")
axes[0].legend()

predicted = []
for index in range(3):
    scale_ratio = (convergence_lmax[0] - 1) * convergence_lmax[0] / ((convergence_lmax[index] - 1) * convergence_lmax[index])
    predicted.append(convergence_errors[0] * scale_ratio**4)
axes[1].loglog(convergence_lmax, convergence_errors, "o-", label="measured")
axes[1].loglog(convergence_lmax, predicted, "k--", label="((L-1) L)^-4")
axes[1].set_xlabel("lmax")
axes[1].set_title("W92 case 2, tilt pi/2: l2(h) after 5 days")
axes[1].legend()
fig.tight_layout()
fig.savefig("figures/benchmark_invariants.png", dpi=110)
plt.show()

## Balance test

The Galewsky jet without its bump is a steady zonal flow, but an unstable one: any non-zonal error grows. `non_zonal_energy` is the energy in the orders $m \neq 0$. It stays at zero here, which shows that the height is in balance with the jet and that the solver creates no non-zonal component.

In [ ]:
sim = simulators["galewsky_unperturbed"]
frames = family_runs["galewsky_unperturbed"]["data"][0]
non_zonal_series = []
for frame in range(frames.shape[0]):
    state = fields_to_spectral(sim.dynamics, frames[frame : frame + 1])
    non_zonal_series.append(non_zonal_energy(state, radius, gravity, 1.0e4).item())
print("non-zonal energy [m2/s2] at days 0 to 6:", non_zonal_series)

zonal_wind = frames[:, :, :, 1].mean(dim=-1)
latitude_degrees = torch.rad2deg(sim.dynamics.latitude[:, 0]).cpu()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(range(len(non_zonal_series)), non_zonal_series, "o-")
axes[0].set_xlabel("day")
axes[0].set_title("unperturbed jet: non-zonal energy [m2/s2]")
axes[1].plot(latitude_degrees, zonal_wind[0], label="day 0")
axes[1].plot(latitude_degrees, zonal_wind[-1], "--", label="day 6")
axes[1].set_xlabel("latitude [deg]")
axes[1].set_title("zonal-mean eastward wind [m/s]")
axes[1].legend()
fig.tight_layout()
fig.savefig("figures/benchmark_balance.png", dpi=110)
plt.show()

## Forced-dissipative regime

With `relaxation_time` the height relaxes towards the balanced height of a zonal jet, and with `drag_time` the wind is damped. The jet that the relaxation rebuilds is unstable, so eddies keep forming and the flow reaches a statistically steady state. Both times are in hours. The eddies persist only with a fast relaxation and a slow drag: 1 day and 100 days here, the values of the D4 configs.

`return_energy_budget=True` returns the energy at every saved frame and the energy change of each process between frames. The run starts from a random balanced flow and lasts 60 days.

In [ ]:
RELAXATION_HOURS = 24.0
DRAG_HOURS = 2400.0

spinup = SphericalShallowWater(
    lmax=lmax, nlat=nlat, nlon=nlon,
    T=60 * 24.0, dt_save=12.0, skip_nt=0,
    batch_size=1, device=device, log_level="warning",
    initial_condition="random_balanced",
    parameters_range={"amp": (15.0, 15.0), "initial_wavenumber": (6.0, 6.0), "initial_bandwidth": (2.0, 2.0)},
    relaxation_time=RELAXATION_HOURS,
    relaxation_jet_speed=80.0,
    drag_time=DRAG_HOURS,
    return_energy_budget=True,
)
spinup_run = spinup.forward_samples_spatiotemporal(n=1, random_seed=0)
spinup_frames = spinup_run["data"][0]
spinup_budget = spinup_run["energy_budget"][0]
print("columns of the energy budget:", spinup.energy_budget_names)

eddy_energy = []
for frame in range(spinup_frames.shape[0]):
    state = fields_to_spectral(spinup.dynamics, spinup_frames[frame : frame + 1])
    eddy_energy.append(non_zonal_energy(state, radius, gravity, 1.0e4).item())

spun_up_state = spinup_frames[-1].clone()
state = fields_to_spectral(spinup.dynamics, spun_up_state.unsqueeze(0))
spun_up_vorticity = spinup.dynamics.to_grid(state[:, 1])[0].cpu()

In [ ]:
spinup_days = torch.arange(spinup_frames.shape[0]) * 0.5
fig, axes = plt.subplots(1, 3, figsize=(19, 4.5))
axes[0].plot(spinup_days, spinup_budget[:, 0])
axes[0].set_xlabel("day")
axes[0].set_title("total energy [m3/s2]")
axes[1].plot(spinup_days, eddy_energy)
axes[1].set_xlabel("day")
axes[1].set_title("non-zonal (eddy) energy [m2/s2]")
limit = spun_up_vorticity.abs().max().item()
axes[2].imshow(spun_up_vorticity, extent=[0, 360, -90, 90], cmap="RdBu_r", vmin=-limit, vmax=limit, aspect="auto")
axes[2].set_title("vorticity at day 60 [1/s]")
axes[2].set_xlabel("longitude [deg]")
fig.tight_layout()
fig.savefig("figures/benchmark_spinup.png", dpi=110)
plt.show()

## Stochastic forcings

`forcing_type` adds a random forcing after every solver step:

- `vortical`: a streamfunction pattern in spherical harmonics, each coefficient an AR(1) process with the decorrelation time `forcing_correlation_time` (Berner et al. 2009). `forcing_energy_rate` is the kinetic energy that it injects per unit mass and time.
- `momentum`: the same with a divergent part, `forcing_divergent_fraction` of the energy.
- `mass_pulses`: Gaussian height bumps at random times and places.
- `forcing_backscatter_fraction` adds a fraction of the diagnosed dissipation to the energy rate. With `forcing_backscatter_mode="local"` the forcing is stronger where the flow dissipates more, so the noise depends on the state. `backscatter_include_drag=True` counts the loss to the drag as dissipation, next to the loss to the hyperdiffusion.

`initial_condition="restart"` starts every run from the spun-up state above. `return_additional_input_fields=True` returns the increments that the forcing added between saved frames. The energy rate printed below is that of the random pattern, so it is zero for the mass pulses, whose effect shows in the forcing work.

In [ ]:
FORCING_ENERGY_RATE = 1.0e-5
FORCING_CORRELATION_HOURS = 6.0

forced_simulators = {}

forced_simulators["none"] = SphericalShallowWater(
    lmax=lmax, nlat=nlat, nlon=nlon,
    T=48.0, dt_save=6.0, skip_nt=0,
    batch_size=1, device=device, log_level="warning",
    initial_condition="restart", initial_state=spun_up_state,
    relaxation_time=RELAXATION_HOURS, drag_time=DRAG_HOURS,
    forcing_type="none",
    return_additional_input_fields=True, return_energy_budget=True,
)

forced_simulators["vortical"] = SphericalShallowWater(
    lmax=lmax, nlat=nlat, nlon=nlon,
    T=48.0, dt_save=6.0, skip_nt=0,
    batch_size=1, device=device, log_level="warning",
    initial_condition="restart", initial_state=spun_up_state,
    relaxation_time=RELAXATION_HOURS, drag_time=DRAG_HOURS,
    forcing_type="vortical",
    forcing_energy_rate=FORCING_ENERGY_RATE,
    forcing_correlation_time=FORCING_CORRELATION_HOURS,
    return_additional_input_fields=True, return_energy_budget=True,
)

forced_simulators["momentum"] = SphericalShallowWater(
    lmax=lmax, nlat=nlat, nlon=nlon,
    T=48.0, dt_save=6.0, skip_nt=0,
    batch_size=1, device=device, log_level="warning",
    initial_condition="restart", initial_state=spun_up_state,
    relaxation_time=RELAXATION_HOURS, drag_time=DRAG_HOURS,
    forcing_type="momentum",
    forcing_energy_rate=FORCING_ENERGY_RATE,
    forcing_correlation_time=FORCING_CORRELATION_HOURS,
    forcing_divergent_fraction=0.5,
    return_additional_input_fields=True, return_energy_budget=True,
)

forced_simulators["local backscatter"] = SphericalShallowWater(
    lmax=lmax, nlat=nlat, nlon=nlon,
    T=48.0, dt_save=6.0, skip_nt=0,
    batch_size=1, device=device, log_level="warning",
    initial_condition="restart", initial_state=spun_up_state,
    relaxation_time=RELAXATION_HOURS, drag_time=DRAG_HOURS,
    forcing_type="vortical",
    forcing_energy_rate=0.0,
    forcing_correlation_time=FORCING_CORRELATION_HOURS,
    forcing_backscatter_fraction=0.2,
    forcing_backscatter_mode="local",
    backscatter_include_drag=True,
    return_additional_input_fields=True, return_energy_budget=True,
)

forced_simulators["mass_pulses"] = SphericalShallowWater(
    lmax=lmax, nlat=nlat, nlon=nlon,
    T=48.0, dt_save=6.0, skip_nt=0,
    batch_size=1, device=device, log_level="warning",
    initial_condition="restart", initial_state=spun_up_state,
    relaxation_time=RELAXATION_HOURS, drag_time=DRAG_HOURS,
    forcing_type="mass_pulses",
    pulse_rate=1.0,
    pulse_amplitude=60.0,
    pulse_width=0.1,
    return_additional_input_fields=True, return_energy_budget=True,
)

forced_runs = {}
for name in forced_simulators:
    forced_runs[name] = forced_simulators[name].forward_samples_spatiotemporal(n=1, random_seed=1)
    budget = forced_runs[name]["energy_budget"][0]  # (frames, 7), columns energy_budget_names
    forcing_work = budget[:-1, 5].sum().item()
    mean_energy_rate = budget[:-1, 6].mean().item()
    print(f"{name:18s} forcing work over 48 h: {forcing_work:10.1f} m3/s2, mean energy rate: {mean_energy_rate:.2e} m2/s3")

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(25, 3.8))
channel = {"none": 1, "vortical": 1, "momentum": 1, "local backscatter": 1, "mass_pulses": 0}
label = {0: "increment of h [m]", 1: "increment of u_phi [m/s]"}
for column, name in enumerate(forced_simulators):
    increments = forced_runs[name]["additional_input_fields"][0, 0, :, :, channel[name]]
    limit = max(increments.abs().max().item(), 1e-12)
    axes[column].imshow(increments, extent=[0, 360, -90, 90], cmap="RdBu_r", vmin=-limit, vmax=limit, aspect="auto")
    axes[column].set_title(name + "\n" + label[channel[name]] + ", first 6 h")
    axes[column].set_xlabel("longitude [deg]")
fig.tight_layout()
fig.savefig("figures/benchmark_forcing_increments.png", dpi=110)
plt.show()

# Kinetic-energy spectrum of the vortical increments, against the power law l^(2p+3)
sim = forced_simulators["vortical"]
increments = forced_runs["vortical"]["additional_input_fields"][:, 0]
state = fields_to_spectral(sim.dynamics, increments)
increment_spectrum = kinetic_energy_spectrum(state, radius)[0, 0].cpu()
slope = 2 * sim.forcing_spectral_slope + 3
reference = increment_spectrum[10] * (degrees[1:] / 10.0) ** slope
fig, axis = plt.subplots(figsize=(6.5, 4.5))
axis.loglog(degrees[1:], increment_spectrum[1:], label="vortical increments of the first 6 h")
axis.loglog(degrees[1:], reference, "k--", label=f"l^(2p+3) = l^{slope:.2f}")
axis.set_xlabel("degree l")
axis.set_title("kinetic energy of the forcing increments [m2/s2]")
axis.legend()
fig.tight_layout()
fig.savefig("figures/benchmark_forcing_spectrum.png", dpi=110)
plt.show()

In [ ]:
budget = forced_runs["vortical"]["energy_budget"][0]
names = forced_simulators["vortical"].energy_budget_names
interval_hours = torch.arange(budget.shape[0] - 1) * 6.0 + 3.0
energy_change = budget[1:, 0] - budget[:-1, 0]
sum_of_terms = budget[:-1, 1:6].sum(dim=1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for column in range(1, 6):
    axes[0].plot(interval_hours, budget[:-1, column], "o-", label=names[column])
axes[0].set_xlabel("hour")
axes[0].set_title("vortical forcing: energy change per 6 h [m3/s2]")
axes[0].legend(fontsize=8)
axes[1].plot(interval_hours, energy_change, "o-", label="change of the total energy")
axes[1].plot(interval_hours, sum_of_terms, "k--", label="sum of the five terms")
axes[1].set_xlabel("hour")
axes[1].set_title("closure of the budget [m3/s2]")
axes[1].legend()
fig.tight_layout()
fig.savefig("figures/benchmark_energy_budget.png", dpi=110)
plt.show()

## Reference ensemble

A restart with `n` simulations gives `n` members from the same state, each with its own forcing: a sample of the true forecast distribution. The forcing pattern has memory, so the convention at the branch point matters. With `branch_forcing_state="redraw"` (the default) the pattern is drawn anew for each member, and the forecast depends on the saved fields alone. With `"carry"` the members inherit the pattern, which an emulator then needs as an extra input (`return_forcing_state=True`).

Here member 0 plays the truth and the 16 others form the ensemble. For a calibrated ensemble the spread matches the error of the ensemble mean, scale by scale.

In [ ]:
ensemble_simulator = SphericalShallowWater(
    lmax=lmax, nlat=nlat, nlon=nlon,
    T=120.0, dt_save=12.0, skip_nt=0,
    batch_size=17, device=device, log_level="warning",
    initial_condition="restart", initial_state=spun_up_state,
    relaxation_time=RELAXATION_HOURS, drag_time=DRAG_HOURS,
    forcing_type="vortical",
    forcing_energy_rate=FORCING_ENERGY_RATE,
    forcing_correlation_time=FORCING_CORRELATION_HOURS,
    branch_forcing_state="redraw",
)
ensemble_run = ensemble_simulator.forward_samples_spatiotemporal(n=17, random_seed=2)
ensemble_frames = ensemble_run["data"]  # (17, frames, nlat, nlon, 3)
truth = ensemble_frames[0]
members = ensemble_frames[1:]
ensemble_mean = members.mean(dim=0)

# Spread of the members and error of their mean, root mean square over the sphere, for u_phi
area_weights = ensemble_simulator.dynamics.quad_weights.cpu().float()
area_weights = area_weights / area_weights.sum()
member_variance = members[..., 1].var(dim=0)
squared_error = (ensemble_mean[..., 1] - truth[..., 1]) ** 2
spread = torch.sqrt((member_variance * area_weights).sum(dim=(1, 2)))
error = torch.sqrt((squared_error * area_weights).sum(dim=(1, 2)))

# Per degree at the last frame: kinetic energy of the member deviations and of the error
deviations = members[:, -1] - ensemble_mean[-1]  # (16, nlat, nlon, 3)
deviation_state = fields_to_spectral(ensemble_simulator.dynamics, deviations)
deviation_spectrum = kinetic_energy_spectrum(deviation_state, radius)  # (16, 2, lmax)
spread_spectrum = deviation_spectrum.sum(dim=1).mean(dim=0).cpu()  # rotational + divergent, mean over members

mean_error = truth[-1] - ensemble_mean[-1]  # (nlat, nlon, 3)
error_state = fields_to_spectral(ensemble_simulator.dynamics, mean_error.unsqueeze(0))
error_spectrum = kinetic_energy_spectrum(error_state, radius).sum(dim=1)[0].cpu()
print("spread of u_phi [m/s] every 12 h:", spread.numpy().round(3))
print("error of the mean [m/s] every 12 h:", error.numpy().round(3))

In [ ]:
lead_days = torch.arange(spread.shape[0]) * 0.5
fig, axes = plt.subplots(1, 3, figsize=(19, 4.5))
axes[0].plot(lead_days, spread, "o-", label="spread of the 16 members")
axes[0].plot(lead_days, error, "s--", label="error of the ensemble mean")
axes[0].set_xlabel("lead time [days]")
axes[0].set_title("u_phi [m/s]: spread and error")
axes[0].legend()
axes[1].loglog(degrees[1:], spread_spectrum[1:], label="members about their mean")
axes[1].loglog(degrees[1:], error_spectrum[1:], "--", label="truth about the mean")
axes[1].set_xlabel("degree l")
axes[1].set_title("kinetic energy per degree at day 5 [m2/s2]")
axes[1].legend()
difference = members[0, -1, :, :, 1] - members[1, -1, :, :, 1]
limit = difference.abs().max().item()
axes[2].imshow(difference, extent=[0, 360, -90, 90], cmap="RdBu_r", vmin=-limit, vmax=limit, aspect="auto")
axes[2].set_title("u_phi of member 1 minus member 2, day 5 [m/s]")
axes[2].set_xlabel("longitude [deg]")
fig.tight_layout()
fig.savefig("figures/benchmark_ensemble.png", dpi=110)
plt.show()

## Coarse-grained truth and physics baseline

`output_lmax` saves the run truncated to a lower band limit, on a coarser grid: the truth as an emulator at that resolution sees it. Here the solver runs at `lmax = 86` and saves at `lmax = 43`. `return_baseline=True` also runs the solver at `lmax = 43` from the truncated first frame. The difference between the two comes from the scales that the low resolution does not resolve, without any added noise.

In [ ]:
coarse_simulator = SphericalShallowWater(
    lmax=86, nlat=128, nlon=256,
    T=144.0, dt_save=12.0, skip_nt=0,
    batch_size=1, device=device, log_level="warning",
    initial_condition="galewsky",
    parameters_range={
        "jet_speed": (80.0, 80.0),
        "height_perturbation_m": (120.0, 120.0),
        "perturbation_lon": (math.pi, math.pi),
        "perturbation_lat": (math.pi / 4, math.pi / 4),
    },
    output_lmax=43, output_nlat=64, output_nlon=128,
    return_baseline=True,
)
coarse_run = coarse_simulator.forward_samples_spatiotemporal(n=1, random_seed=0)
coarse_truth = coarse_run["data"][0]
baseline = coarse_run["baseline"][0]
print("coarse-grained truth:", tuple(coarse_truth.shape), "baseline:", tuple(baseline.shape))

low = coarse_simulator.output_dynamics
baseline_errors = []
for frame in range(coarse_truth.shape[0]):
    height_truth = coarse_truth[frame, :, :, 0].double().to(device) + 1.0e4
    height_baseline = baseline[frame, :, :, 0].double().to(device) + 1.0e4
    l1, l2, linf = height_error_norms(height_baseline, height_truth, low.quad_weights)
    baseline_errors.append(l2.item())

state = fields_to_spectral(low, coarse_truth[-1:])
vorticity_truth = low.to_grid(state[:, 1])[0].cpu()
state = fields_to_spectral(low, baseline[-1:])
vorticity_baseline = low.to_grid(state[:, 1])[0].cpu()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 4.5))
limit = vorticity_truth.abs().max().item()
axes[0].imshow(vorticity_truth, extent=[0, 360, -90, 90], cmap="RdBu_r", vmin=-limit, vmax=limit, aspect="auto")
axes[0].set_title("coarse-grained truth (lmax 86 saved at 43)\nvorticity at day 6 [1/s]")
axes[1].imshow(vorticity_baseline, extent=[0, 360, -90, 90], cmap="RdBu_r", vmin=-limit, vmax=limit, aspect="auto")
axes[1].set_title("baseline (solver at lmax 43)\nvorticity at day 6 [1/s]")
axes[2].semilogy(torch.arange(len(baseline_errors)) * 0.5, baseline_errors, "o-")
axes[2].set_xlabel("lead time [days]")
axes[2].set_title("l2(h) of the baseline against the truth")
fig.tight_layout()
fig.savefig("figures/benchmark_coarse_graining.png", dpi=110)
plt.show()

## Rotation

`rotate_fields` rotates saved frames by an element of SO(3), given by three angles: the rotation is $R_z(\alpha) R_y(\beta) R_z(\gamma)$. The height is a scalar field. The wind is a spin-1 field: its components are measured along the local east and south directions, which the rotation changes, so the two maps cannot be rotated as two scalars.

W92 case 2 gives an exact check. The flow about the geographic axis, rotated by $\beta = -\text{tilt}$, must be the analytic flow about the tilted axis.

In [ ]:
tilt = 0.8
straight_simulator = SphericalShallowWater(
    lmax=lmax, nlat=nlat, nlon=nlon,
    T=0.0, dt_save=24.0, skip_nt=0,
    batch_size=1, device=device, log_level="warning",
    initial_condition="williamson2",
    mean_depth=williamson_depth,
    parameters_range={"flow_speed": (38.61, 38.61), "tilt": (0.0, 0.0)},
)
tilted_simulator = SphericalShallowWater(
    lmax=lmax, nlat=nlat, nlon=nlon,
    T=0.0, dt_save=24.0, skip_nt=0,
    batch_size=1, device=device, log_level="warning",
    initial_condition="williamson2",
    mean_depth=williamson_depth,
    parameters_range={"flow_speed": (38.61, 38.61), "tilt": (tilt, tilt)},
)
straight_run = straight_simulator.forward_samples_spatiotemporal(n=1, random_seed=0)
tilted_run = tilted_simulator.forward_samples_spatiotemporal(n=1, random_seed=0)
straight_frame = straight_run["data"][:, 0]  # (1, nlat, nlon, 3)
tilted_frame = tilted_run["data"][:, 0]

rotated_frame = rotate_fields(straight_simulator.dynamics, straight_frame, 0.0, -tilt, 0.0)
rotated_frame = rotated_frame.float().cpu()
rotation_error = (rotated_frame - tilted_frame).abs().amax(dim=(0, 1, 2))
print(f"largest error of the rotated frame: h {rotation_error[0]:.1e} m, u_phi {rotation_error[1]:.1e} m/s, u_theta {rotation_error[2]:.1e} m/s")

# The wrong way: the map of u_phi rotated as if it were a scalar field
dynamics = straight_simulator.dynamics
u_phi_map = straight_frame[..., 1].double().to(device)
u_phi_coefficients = dynamics.to_coefficients(u_phi_map)
rotated_coefficients = rotate_coefficients(u_phi_coefficients, 0.0, -tilt, 0.0)
rotated_as_scalar = dynamics.to_grid(rotated_coefficients).float().cpu()
scalar_error = (rotated_as_scalar - tilted_frame[..., 1]).abs().max().item()
print(f"largest error of u_phi rotated as a scalar: {scalar_error:.1f} m/s")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 4))
panel_fields = [
    straight_frame[0, :, :, 1],
    rotated_frame[0, :, :, 1],
    tilted_frame[0, :, :, 1],
    rotated_as_scalar[0],
]
panel_titles = [
    "u_phi about the geographic axis [m/s]",
    "rotated as a spin-1 field",
    "analytic flow about the tilted axis",
    "u_phi rotated as a scalar (wrong)",
]
for column in range(4):
    axes[column].imshow(panel_fields[column], extent=[0, 360, -90, 90], cmap="RdBu_r", vmin=-40, vmax=40, aspect="auto")
    axes[column].set_title(panel_titles[column])
    axes[column].set_xlabel("longitude [deg]")
fig.tight_layout()
fig.savefig("figures/benchmark_rotation.png", dpi=110)
plt.show()